# 04 — Evaluation & Results

We compare all imputation methods across:
- **Datasets**: California Housing, Adult Census
- **Mechanisms**: MCAR, MAR, MNAR
- **Metrics**: RMSE, MAE, KS test, Downstream ML performance

This notebook produces the main findings of the study.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import sys
sys.path.append('..')
from src.missing_generator import introduce_mcar, introduce_mar, introduce_mnar
from src.evaluation import (
    evaluate_imputation,
    downstream_regression,
    downstream_classification,
    plot_distribution_comparison,
    plot_metrics_comparison,
    plot_missing_heatmap,
)

sns.set_theme(style='whitegrid')
%matplotlib inline
SEED = 42

## 1. Load Data

In [ ]:
with open('../data/processed/housing_imputed.pkl', 'rb') as f:
    housing_imputed = pickle.load(f)

with open('../data/processed/adult_imputed.pkl', 'rb') as f:
    adult_imputed = pickle.load(f)

housing_original = pd.read_csv('../data/processed/housing_test_original.csv')
adult_original   = pd.read_csv('../data/processed/adult_test_original.csv')

print('Data loaded.')

## 2. Rebuild Missing Masks

We need to know which positions were originally NaN to evaluate only at those positions.

In [ ]:
housing_target_cols = ['MedInc', 'HouseAge', 'AveRooms']

housing_masks = {
    'MCAR': introduce_mcar(housing_original, columns=housing_target_cols, missing_rate=0.3).isna(),
    'MAR':  introduce_mar(housing_original,  target_col='MedInc', condition_col='HouseAge', missing_rate=0.3).isna(),
    'MNAR': introduce_mnar(housing_original, target_col='MedInc', missing_rate=0.3, direction='high').isna(),
}

adult_target_cols = ['age', 'hours_per_week', 'capital_gain']

adult_masks = {
    'MCAR': introduce_mcar(adult_original, columns=adult_target_cols, missing_rate=0.3).isna(),
    'MAR':  introduce_mar(adult_original,  target_col='hours_per_week', condition_col='age', missing_rate=0.3).isna(),
    'MNAR': introduce_mnar(adult_original, target_col='capital_gain', missing_rate=0.3, direction='high').isna(),
}

## 3. Imputation Quality Metrics — California Housing

In [ ]:
housing_results = []
for mechanism in ['MCAR', 'MAR', 'MNAR']:
    for method_name, df_imp in housing_imputed[mechanism].items():
        metrics_df = evaluate_imputation(housing_original, df_imp, housing_masks[mechanism])
        metrics_df['method'] = method_name
        metrics_df['mechanism'] = mechanism
        housing_results.append(metrics_df)

housing_results = pd.concat(housing_results, ignore_index=True)
housing_results.head(12)

In [ ]:
for mechanism in ['MCAR', 'MAR', 'MNAR']:
    subset = housing_results[housing_results['mechanism'] == mechanism]
    plot_metrics_comparison(subset, metric='rmse')
    plt.suptitle(f'Housing — RMSE by method ({mechanism})', y=1.01)

## 4. Imputation Quality Metrics — Adult Census

In [ ]:
adult_results = []
for mechanism in ['MCAR', 'MAR', 'MNAR']:
    for method_name, df_imp in adult_imputed[mechanism].items():
        metrics_df = evaluate_imputation(adult_original, df_imp, adult_masks[mechanism])
        metrics_df['method'] = method_name
        metrics_df['mechanism'] = mechanism
        adult_results.append(metrics_df)

adult_results = pd.concat(adult_results, ignore_index=True)
adult_results.head(12)

In [ ]:
for mechanism in ['MCAR', 'MAR', 'MNAR']:
    subset = adult_results[adult_results['mechanism'] == mechanism]
    plot_metrics_comparison(subset, metric='rmse')
    plt.suptitle(f'Adult — RMSE by method ({mechanism})', y=1.01)

## 5. Distribution Comparison

In [ ]:
for mechanism in ['MCAR', 'MAR', 'MNAR']:
    imputed_dict = {name: df['MedInc'] for name, df in housing_imputed[mechanism].items()}
    plot_distribution_comparison(housing_original['MedInc'], imputed_dict, col_name=f'MedInc ({mechanism})')

## 6. Downstream Evaluation — California Housing (Regression)

In [ ]:
downstream_housing = []
baseline_rmse = downstream_regression(housing_original, target_col='MedHouseVal', seed=SEED)
print(f'Baseline RMSE (no missing): {baseline_rmse:.4f}')

for mechanism in ['MCAR', 'MAR', 'MNAR']:
    for method_name, df_imp in housing_imputed[mechanism].items():
        rmse = downstream_regression(df_imp, target_col='MedHouseVal', seed=SEED)
        downstream_housing.append({'method': method_name, 'mechanism': mechanism, 'rmse': rmse})

downstream_housing = pd.DataFrame(downstream_housing)
downstream_housing

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
downstream_housing.pivot(index='method', columns='mechanism', values='rmse').plot(kind='bar', ax=ax)
ax.axhline(baseline_rmse, color='black', linestyle='--', label='Baseline (no missing)')
ax.set_title('Downstream RMSE — California Housing')
ax.set_ylabel('RMSE')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 7. Downstream Evaluation — Adult Census (Classification)

In [ ]:
downstream_adult = []
baseline_acc = downstream_classification(adult_original, target_col='income', seed=SEED)
print(f'Baseline Accuracy (no missing): {baseline_acc:.4f}')

for mechanism in ['MCAR', 'MAR', 'MNAR']:
    for method_name, df_imp in adult_imputed[mechanism].items():
        acc = downstream_classification(df_imp, target_col='income', seed=SEED)
        downstream_adult.append({'method': method_name, 'mechanism': mechanism, 'accuracy': acc})

downstream_adult = pd.DataFrame(downstream_adult)
downstream_adult

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
downstream_adult.pivot(index='method', columns='mechanism', values='accuracy').plot(kind='bar', ax=ax)
ax.axhline(baseline_acc, color='black', linestyle='--', label='Baseline (no missing)')
ax.set_title('Downstream Accuracy — Adult Census')
ax.set_ylabel('Accuracy')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 8. Summary Table

In [ ]:
print('=== Housing — Mean RMSE per method and mechanism ===')
print(housing_results.groupby(['method', 'mechanism'])['rmse'].mean().unstack().round(4))

print('\n=== Adult — Mean RMSE per method and mechanism (numerical cols) ===')
num_adult = adult_results[adult_results['type'] == 'numerical']
print(num_adult.groupby(['method', 'mechanism'])['rmse'].mean().unstack().round(4))